# Parkinson Ses Verileri ile Makine Öğrenmesi Sınıflandırma Projesi

**Amaç:** UCI Machine Learning Repository'deki Parkinson ses ölçümleri veri setini kullanarak `status` sütununu tahmin eden sınıflandırma modelleri kurmak.

> **Önemli:** Bu çalışma eğitim amaçlıdır. Tıbbi tanı koymaz ve gerçek sağlık kararlarında kullanılamaz.

## Veri seti hakkında
UCI kaynağındaki Oxford Parkinson's Disease Detection Dataset, 31 kişiden alınan 195 ses kaydını ve biyomedikal ses ölçümlerini içerir. `status` hedef sütununda `1` Parkinson hastalığı grubunu, `0` sağlıklı kontrol grubunu belirtir. Veri kaynağı ve atıf bilgisi:
- https://archive.ics.uci.edu/dataset/174/parkinsons
- Little, M. (2007). *Parkinsons*. UCI Machine Learning Repository. https://doi.org/10.24432/C59C74

Bu veri seti küçük ve belirli bir örneklemden oluştuğu için sonuçlar genellenebilir klinik performans olarak yorumlanmamalıdır.


## 1. Gerekli kütüphaneler

Aşağıdaki hücre, veri analizi, görselleştirme ve makine öğrenmesi için gerekli paketleri yükler.

In [ ]:
!pip -q install ucimlrepo

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

RANDOM_STATE = 42
print("Kütüphaneler hazır.")

## 2. Veri setini yükleme

Veri, UCI kaynağından `ucimlrepo` paketiyle alınır. İnternet bağlantısı gerekir.

In [ ]:
dataset = fetch_ucirepo(id=174)

X = dataset.data.features.copy()
y = dataset.data.targets.copy()

# Hedef sütununu tek boyutlu hale getir
if isinstance(y, pd.DataFrame):
    y = y.iloc[:, 0]

print("Özellik boyutu:", X.shape)
print("Hedef boyutu:", y.shape)
display(X.head())

## 3. Veriyi tanıma ve kontrol etme

Sütun türleri, eksik değerler ve hedef sınıf dağılımı incelenir.

In [ ]:
display(X.info())
display(X.describe().T)
print("Eksik değer sayıları:")
display(X.isnull().sum())
print("Hedef sınıf dağılımı:")
display(y.value_counts().sort_index())

In [ ]:
plt.figure(figsize=(5, 4))
sns.countplot(x=y)
plt.title("Hedef sınıf dağılımı (status)")
plt.xlabel("status (0: kontrol, 1: Parkinson grubu)")
plt.ylabel("Kayıt sayısı")
plt.tight_layout()
plt.show()

## 4. Veri hazırlama ve eğitim/test ayırma

`name` sütunu kişi/kayıt kimliği olduğu için model özelliği olarak kullanılmaz. Sayısal ses ölçümleri kullanılır. Veri, sınıf oranlarını koruyacak şekilde eğitim ve test kümelerine ayrılır.

In [ ]:
X_model = X.copy()

# Kimlik sütununu varsa çıkar
if "name" in X_model.columns:
    X_model = X_model.drop(columns=["name"])

# Sayısal olmayan sütunları çıkar; bu veri setinde ölçümler sayısaldır
X_model = X_model.select_dtypes(include=[np.number])

X_train, X_test, y_train, y_test = train_test_split(
    X_model, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print("Eğitim:", X_train.shape)
print("Test:", X_test.shape)

## 5. Modelleri eğitme ve karşılaştırma

Üç sınıflandırma modeli denenir:
- Lojistik Regresyon
- Destek Vektör Makineleri (SVM)
- Rastgele Orman (Random Forest)

Ölçeklendirme gereken modellerde `StandardScaler`, veri sızıntısını önlemek için `Pipeline` içinde kullanılır.

In [ ]:
models = {
    "Lojistik Regresyon": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000, class_weight="balanced"))
    ]),
    "SVM": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", SVC(class_weight="balanced"))
    ]),
    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=300, random_state=RANDOM_STATE,
            class_weight="balanced"
        ))
    ])
}

results = []
trained_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    trained_models[name] = model

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1-score": f1_score(y_test, predictions, zero_division=0)
    })

results_df = pd.DataFrame(results).sort_values("F1-score", ascending=False)
display(results_df)

## 6. En iyi modelin ayrıntılı değerlendirmesi

Bu bölümde test kümesinde F1-score değeri en yüksek çıkan modelin sınıflandırma raporu ve karmaşıklık matrisi gösterilir.

In [ ]:
best_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_name]
best_predictions = best_model.predict(X_test)

print("Seçilen model:", best_name)
print(classification_report(y_test, best_predictions, zero_division=0))

cm = confusion_matrix(y_test, best_predictions)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title(f"Karmaşıklık Matrisi - {best_name}")
plt.xlabel("Tahmin")
plt.ylabel("Gerçek")
plt.tight_layout()
plt.show()

## 7. Sonuçları yorumlama

Aşağıdaki hücre, model karşılaştırma tablosunu yeniden gösterir. Rapor yazarken yalnızca burada elde ettiğiniz gerçek değerleri kullanın.

- **Accuracy:** Tüm tahminlerin ne kadarının doğru olduğunu gösterir.
- **Precision:** Modelin pozitif dediği kayıtların ne kadarının gerçekten pozitif olduğunu gösterir.
- **Recall:** Gerçek pozitif kayıtların ne kadarının yakalandığını gösterir.
- **F1-score:** Precision ve Recall değerlerini birlikte özetler.

**Dikkat edilmesi gereken sınırlılık:** Aynı kişiden birden fazla ses kaydı bulunduğundan, rastgele kayıt bazlı eğitim/test ayrımı aynı kişiye ait kayıtları iki kümeye de koyabilir. Bu durum performansın olduğundan yüksek görünmesine yol açabilir. Bu nedenle sonuçlar dikkatli yorumlanmalıdır. Daha ileri bir çalışmada kişi bazlı grup ayırma (Group split) tercih edilebilir.

## 8. Proje sonuç özeti

Aşağıdaki hücre, rapora aktarabilmeniz için en iyi modelin metriklerini yazdırır.

In [ ]:
best_row = results_df.iloc[0]
print(f"En yüksek F1-score değerine sahip model: {best_row['Model']}")
print(f"Accuracy : {best_row['Accuracy']:.3f}")
print(f"Precision: {best_row['Precision']:.3f}")
print(f"Recall   : {best_row['Recall']:.3f}")
print(f"F1-score : {best_row['F1-score']:.3f}")

print("\nGenel değerlendirme:")
print("Bu proje, ses ölçümlerinden yararlanarak sınıflandırma modellerinin")
print("eğitim ve test performansını karşılaştırmıştır. Bu sonuçlar tıbbi tanı değildir.")

## Kaynakça

Little, M. (2007). *Parkinsons*. UCI Machine Learning Repository. https://doi.org/10.24432/C59C74

UCI veri seti sayfası: https://archive.ics.uci.edu/dataset/174/parkinsons
